# Nuclei segmentation




Implement some deconvolution code

In [1]:
from typing import List

import math
import numpy as np
from scipy.special import jn


def compute_otf(
    shape: List[int],
    spacing: List[float],
    numerical_aperture: float,
    medium_refactive_index: float,
    wavelength: float,
):
    """Basic defocus optical transfer function for wide field

    Parameters
    ----------
    shape : list[int]
        image size [D,H,W]
    spacing: list [float]
        spacing [D,H,W] in 3D in physical unit
    numerical_aperture: float
        objective numerical aperture
    medium_refractive_index: float
        the refractive index of the medium
    wavelength: float
        the emission wavelength in unit

    Returns
    -------
    otf : np.ndarray(dtype=np.complex128)
        The optical transfer function
    """
    a = (numerical_aperture / wavelength) ** 2
    b = (medium_refactive_index / wavelength) ** 2
    kx = np.fft.fftfreq(shape[2], spacing[2]).reshape([1, 1, shape[2]])
    ky = np.fft.fftfreq(shape[1], spacing[1]).reshape([1, shape[1], 1])
    z = (
        np.concatenate(
            (np.arange(0, shape[0] // 2), np.arange(-shape[0] // 2, 0))
        ).reshape([shape[0], 1, 1])
        * spacing[0]
    )
    d2 = kx**2 + ky**2
    W = z * np.sqrt(np.maximum(0, b - d2))
    P = (d2 < a).astype(float)
    psf = np.square(np.abs(np.fft.ifft2(P * np.exp(2j * math.pi * W))))
    psf = psf / psf.sum()
    otf = np.fft.fftn(psf)
    return otf


def compute_pinhole_otf(shape, spacing, diameter):
    """Fourier transform of a circular aperture

    $(2 J_1(k) / k)$
    Note
    ----
    THis is not the Airy disk
    """
    kx = np.fft.fftfreq(shape[2], spacing[2]).reshape([1, 1, shape[2]])
    ky = np.fft.fftfreq(shape[1], spacing[1]).reshape([1, shape[1], 1])
    k = math.pi / 2 * diameter * np.sqrt(kx**2 + ky**2)
    plane = np.zeros([1, shape[1], shape[2]])
    plane[k > 1e-8] = 2 * jn(1, k[k > 1e-8]) / k[k > 1e-8]
    plane[0, 0, 0] = 1
    pinhole = np.zeros(shape)
    pinhole[0] = plane
    return pinhole


def compute_confocal_otf(
    shape,
    spacing,
    numerical_aperture=1.4,
    medium_refractive_index=1.35,
    wavelength_ex=0.5,
    wavelength_em=0.5,
    pinhole=0.5,
):
    """Compute the optical transfer function of a confocal microscope"""
    otf1 = compute_otf(
        shape, spacing, numerical_aperture, medium_refractive_index, wavelength_ex
    )
    otf2 = compute_otf(
        shape, spacing, numerical_aperture, medium_refractive_index, wavelength_em
    )
    pinhole = compute_pinhole_otf(shape, spacing, pinhole)
    psf1 = np.real(np.fft.ifftn(otf1 * pinhole))
    psf2 = np.real(np.fft.ifftn(otf2))
    otf = np.fft.fftn(psf1 * psf2)
    return otf


def deconvolve_richardson_lucy_heavy_ball(data, otf, background=0, iterations=100):
    """Deconvolve data according to the given otf using a scaled heavy ball
    Richardson-Lucy algorithm

    Parameters
    ----------
    data       : numpy array
    otf        : numpy array of the same size than data
    iterations : number of iterations

    Returns
    -------
    estimate   : estimated image
    dkl        : the kullback leibler divergence (should tend to 1/2)

    Note
    ----
    [1] H. Wang and P. C. Miller, Scaled Heavy-Ball Acceleration of the
        Richardson-Lucy Algorithm for 3D Microscopy Image Restoration, IEEE
        Transactions on Image Processing, vol. 23, no. 2, pp. 848-854, Feb. 2014,
        doi: 10.1109/TIP.2013.2291324.
    """
    epsilon = 1e-6
    old_estimate = np.maximum(
        np.real(np.fft.ifftn(otf * np.fft.fftn(data - background))), epsilon
    )
    estimate = data
    dkl = np.zeros(iterations)
    for k in range(iterations):
        beta = (k - 1.0) / (k + 2.0)
        prediction = estimate + beta * (estimate - old_estimate)
        blurred = np.maximum(
            np.real(np.fft.ifftn(otf * np.fft.fftn(prediction + background))), epsilon
        )
        ratio = data / blurred
        gradient = 1.0 - np.real(np.fft.ifftn(otf * np.fft.fftn(ratio)))
        old_estimate = estimate
        estimate = np.maximum(prediction - estimate * gradient, 0)
        dkl[k] = np.mean(blurred - data + data * np.log(np.maximum(ratio, epsilon)))
    return estimate, dkl

Load the files from tiff

In [2]:
import tifffile
from pathlib import Path
import numpy as np
import scipy.ndimage as ndi

# TODO: adapt the folder path here
folder = Path("C:/Users/Amy Courtney/Documents/Temp/Marie_HCR_Test")
img_file = folder / "C4-3-01-Stitching-26_Nuclei_Crop.tif"

img = tifffile.imread(img_file)
img = img[:, :, 0:500].astype(float)


## Segmentation of the deconvolved image

In [3]:
from cellpose import models

# List of all available pretrained model types
available_models = models.MODEL_NAMES
print("Available Cellpose models:")
for m in available_models:
    print("-", m)


Available Cellpose models:
- cyto3
- nuclei
- cyto2_cp3
- tissuenet_cp3
- livecell_cp3
- yeast_PhC_cp3
- yeast_BF_cp3
- bact_phase_cp3
- bact_fluor_cp3
- deepbacs_cp3
- cyto2
- cyto
- transformer_cp3
- neurips_cellpose_default
- neurips_cellpose_transformer
- neurips_grayscale_cyto2


In [14]:
from cellpose import models, core
use_gpu = core.use_gpu()
model = models.CellposeModel(gpu=use_gpu, model_type="cyto2_cp3")
masks, flows, styles = model.eval(
    img,
    diameter=22,                 # tune as needed
    flow_threshold=0.4,
    cellprob_threshold=0.0,
)


In [15]:
import napari

v = napari.view_image(img, scale=[0.291, 0.1, 0.1])
v.add_labels(cell_labels_dec, scale=[0.291, 0.1, 0.1])

<Labels layer 'cell_labels_dec' at 0x225e1f998d0>